# 01 · Parquet basics — the file format under every lakehouse

**Goal:** understand what's inside a Parquet file, why engines like it, and what it *can't* do on its own — which is exactly the gap Iceberg fills in notebook 02.

**Databricks link:** every Delta or Iceberg table on Databricks is a folder of Parquet files plus metadata. When a query is fast or slow, Parquet is usually why.

**New to Spark or notebooks?** Do `00_spark_basics.ipynb` first — this notebook assumes you know what a DataFrame and `spark.sql(...)` are.

**How to use this notebook:** run each cell in order with **Shift + Enter**. Before each code cell, a **What this cell does** note explains the code line by line. After each result, read the **What to notice** note below it. Takes about 30 minutes.

Data: NYC Yellow Taxi trips, January 2024 (≈3 million rows), published by NYC TLC as a Parquet file.

> **SQL or Python?** Both. PySpark is the Python program that runs Spark. Most cells use Spark's **DataFrame API** (Python methods like `.filter()`, `.groupBy()`); anything can also be written as SQL with `spark.sql("SELECT ...")`. The **Coming from Hive/Impala** notes show the SQL for the DataFrame lines.

**What this cell does** (setup — new to Spark? do notebook `00_spark_basics` first)
- `import` lines load libraries: `get_spark` (starts Spark), `F` (Spark's SQL functions for Python), `pq` (**PyArrow**, a library that can open Parquet files directly, without Spark), `pd` (**pandas**, for small tables that display nicely), `os` (file and folder operations).
- `spark = get_spark(...)` starts Spark — your session, like an open impala-shell.
- `RAW` and `PLAIN` are variables holding folder paths, so later cells don't repeat them. `/home/lab/work/data` inside the container is the `data/` folder of the repo on your Mac.
- `os.path.getsize(RAW) / 1e6` = the file size in bytes divided by one million → MB.

In [ ]:
from lab_helpers import get_spark
from pyspark.sql import functions as F
import pyarrow.parquet as pq
import pandas as pd
import os

spark = get_spark("01-parquet-basics")
RAW = "/home/lab/work/data/raw/yellow_tripdata_2024-01.parquet"
PLAIN = "/home/lab/work/data/plain"       # where this notebook writes its own Parquet
print("Spark", spark.version, "| file size:", round(os.path.getsize(RAW) / 1e6, 1), "MB")

## 1. What's inside a Parquet file

A Parquet file is **columnar**: values of one column are stored together, not row by row.

```
yellow_tripdata_2024-01.parquet
├── Row group 0  (~1M rows)
│   ├── column chunk: VendorID            (compressed)
│   ├── column chunk: tpep_pickup_datetime
│   └── ... one chunk per column
├── Row group 1
├── Row group 2
└── Footer: schema + where every chunk is + (optional) min/max stats per chunk
```

Engines read the **footer first**, then fetch only the column chunks a query needs. We'll read the footer directly with PyArrow — no Spark involved.

> **Coming from Hive/Impala:** these are the same Parquet files you get from `CREATE TABLE ... STORED AS PARQUET`. Impala reads them the same way: footer first, then only the needed columns. Nothing here is new to the format — what's new is what Iceberg puts *around* these files (notebook 02).

**What this cell does**
- `pq.ParquetFile(RAW)` opens the file with PyArrow and reads **only its footer** (the metadata at the end of the file).
- `meta.num_rows`, `meta.num_row_groups`, ... print values stored in that footer. No data rows are read.
- `print(f.schema_arrow)` prints the column names and types written in the file.

In [ ]:
f = pq.ParquetFile(RAW)
meta = f.metadata
print("rows:        ", meta.num_rows)
print("row groups:  ", meta.num_row_groups)
print("columns:     ", meta.num_columns)
print("written by:  ", meta.created_by)
print()
print(f.schema_arrow)

**What to notice**
- The schema (names *and* types) is in the file itself. Nobody has to tell Spark what the columns are.
- ~3M rows are split into 3 **row groups** — the unit an engine can skip.
- `tpep_pickup_datetime` is a `timestamp` **without** a time zone. Spark calls this `timestamp_ntz`.

**What this cell does** — builds a small report about the first row group, one line per column
- `rg = meta.row_group(0)` picks row group number 0 (Python counts from 0).
- `for i in range(rg.num_columns):` repeats the indented lines once per column (`range(19)` = 0, 1, ... 18).
- For each column `c` it records the compression codec, compressed vs uncompressed size in KB, their ratio, and whether min/max statistics exist — all read from the footer.
- `rows.append({...})` adds one record (a Python `dict`: name → value) to the list `rows`.
- `pd.DataFrame(rows)` turns the list into a pandas table, which the notebook displays as a grid. (This is pandas, not Spark — fine here because it's only 19 rows.)

In [ ]:
# Per-column details for row group 0: compression, sizes, and whether min/max stats exist.
rg = meta.row_group(0)
rows = []
for i in range(rg.num_columns):
    c = rg.column(i)
    rows.append({
        "column": c.path_in_schema,
        "codec": c.compression,
        "compressed_KB": round(c.total_compressed_size / 1024),
        "raw_KB": round(c.total_uncompressed_size / 1024),
        "ratio": round(c.total_uncompressed_size / c.total_compressed_size, 1),
        "has_min_max": c.is_stats_set,
    })
pd.DataFrame(rows)

**What to notice**
- Every column is compressed with **ZSTD**. Repetitive columns (`VendorID`, `payment_type`, `mta_tax`) compress far better than unique ones (timestamps, `trip_distance`).
- `has_min_max` is **False for every column**. The publisher wrote this file *without statistics*. Without min/max per row group, an engine can't skip row groups — it must read them all. Keep this in mind for section 4.

> **Coming from Hive/Impala:** don't confuse these with `COMPUTE STATS`. Impala's `COMPUTE STATS` stores table/column statistics in the **metastore** for the query planner. The stats here live **inside each Parquet file** and decide which row groups a scan can skip. Impala uses both kinds.

## 2. Reading it with Spark

**What this cell does**
- `spark.read.parquet(RAW)` creates a Spark **DataFrame** over the file — a table you can query. Nothing is read yet except the footer (Spark is *lazy*: see notebook 00, section 5).
- `df.printSchema()` = `DESCRIBE`: column names, types, and whether NULLs are allowed.

In [ ]:
df = spark.read.parquet(RAW)
df.printSchema()

**What this cell does**
- `%%time` is a Jupyter feature that measures how long the cell takes.
- `df.count()` = `SELECT count(*) FROM df`. It's an **action**, so Spark actually runs it.

In [ ]:
%%time
df.count()

**What to notice**
- `printSchema()` was instant: Spark read only the footer.
- `count()` is fast too: the row count of each row group is in the footer, so Spark doesn't scan the data.

> **Coming from Hive/Impala:** `spark.read.parquet(path)` is like querying an `EXTERNAL TABLE` over an HDFS folder — without needing `CREATE EXTERNAL TABLE` first, because the schema comes from the files. To use SQL, register it as a view:
> ```python
> df.createOrReplaceTempView("trips")
> spark.sql("SELECT count(*) FROM trips").show()
> ```

## 3. Columnar reads: only fetch what the query needs

`explain()` shows Spark's plan. Look for two lines in the output:
- **`ReadSchema`** — the columns Spark will actually read from disk.
- **`PushedFilters`** — filters handed to the Parquet reader so it can skip data early.

**What this cell does**
- Builds a query equal to `SELECT tpep_pickup_datetime, fare_amount FROM df WHERE fare_amount > 100` using DataFrame methods. `F.col("fare_amount") > 100` is the condition; writing `.filter("fare_amount > 100")` would mean the same.
- The outer `( ... )` only lets the statement span two lines.
- `.explain()` prints Spark's **physical plan** — how it will run the query — without running it.

In [ ]:
expensive = (df.select("tpep_pickup_datetime", "fare_amount")
               .filter(F.col("fare_amount") > 100))
expensive.explain()

**What to notice**
- `ReadSchema` lists only **2 of 19 columns**. The other 17 are never read — that's the columnar win.
- `PushedFilters: [IsNotNull(fare_amount), GreaterThan(fare_amount,100.0)]`. The filter is pushed down, but it can only *skip* row groups if min/max stats exist. This file has none, so every row group is still read and filtered row by row.

> **Coming from Hive/Impala:** `explain()` is Spark's `EXPLAIN`. The DataFrame above is the same as `SELECT tpep_pickup_datetime, fare_amount FROM trips WHERE fare_amount > 100` — you'd get the identical plan from `spark.sql("EXPLAIN SELECT ...")`. `ReadSchema` is the column list Impala's scanner would read; `PushedFilters` are the predicates pushed into the scan, like Impala's Parquet predicate pushdown.

## 4. Write your own Parquet — with statistics

Spark writes min/max stats by default. Let's write the same data ourselves and compare.

**What this cell does**
- `df.repartition(4)` splits the data into 4 chunks (partitions), so the write produces **4 files** — one per chunk.
- `.write.mode("overwrite").parquet(path)` writes Parquet files into the folder `path`, replacing anything already there. (Like `INSERT OVERWRITE DIRECTORY`.)
- `os.listdir(folder)` lists the folder; the `sorted(p for p in ... if p.endswith(".parquet"))` part keeps only the Parquet file names, in order.
- The last lines open the first file with PyArrow and read `fare_amount`'s statistics from its footer — the thing the TLC file didn't have.

In [ ]:
(df.repartition(4)                      # 4 output files
   .write.mode("overwrite")
   .parquet(f"{PLAIN}/trips_jan"))

files = sorted(p for p in os.listdir(f"{PLAIN}/trips_jan") if p.endswith(".parquet"))
print(len(files), "files:", files[0], "...")

ours = pq.ParquetFile(f"{PLAIN}/trips_jan/{files[0]}")
col = ours.metadata.row_group(0).column(ours.schema_arrow.names.index("fare_amount"))
print("codec:", col.compression, "| stats set:", col.is_stats_set,
      "| fare_amount min/max in row group 0:", col.statistics.min, "/", col.statistics.max)

**What to notice**
- Spark wrote **Snappy** compression (Spark's default) instead of ZSTD — a setting, not a law.
- Now each row group has **min/max** stats. A query like `fare_amount > 5000` could skip any row group whose max is below 5000 without reading it.
- `_SUCCESS` (in the folder) is how plain Parquet says "the write finished". It's just an empty marker file — readers aren't forced to check it.

## 5. Partitioning: one folder per value

Partitioning splits data into folders by a column's value (Hive style: `pickup_date=2024-01-15/`). A query that filters on that column reads only the matching folders.

> **Coming from Hive/Impala:** this is exactly `PARTITIONED BY (pickup_date STRING)` in Hive. The write below is the same as `INSERT OVERWRITE TABLE t PARTITION (pickup_date) SELECT ..., to_date(tpep_pickup_datetime) FROM trips` with dynamic partitioning.

**What this cell does**
- `df.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))` adds a new column = `to_date(tpep_pickup_datetime) AS pickup_date`, keeping all existing columns.
- `.write.partitionBy("pickup_date")` writes one **folder per distinct value**: `pickup_date=2024-01-15/`, ... — exactly Hive-style partitioning.
- The rest lists the folders and counts the `.parquet` files inside them. `sum(len(...) for d in folders)` = "for each folder count its files, then add up".

In [ ]:
by_day = df.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
by_day.write.mode("overwrite").partitionBy("pickup_date").parquet(f"{PLAIN}/trips_by_day")

folders = sorted(d for d in os.listdir(f"{PLAIN}/trips_by_day") if d.startswith("pickup_date="))
n_files = sum(len([f for f in os.listdir(f"{PLAIN}/trips_by_day/{d}") if f.endswith(".parquet")]) for d in folders)
print(len(folders), "folders,", n_files, "files")
print("first folders:", folders[:4])
print("last folders: ", folders[-3:])

**What to notice**
- You expected 31 folders for January — you got more. The data has trips dated **2002, 2009, Dec 2023 and Feb 2024** (bad clock data from taxi meters). Each odd value becomes its own folder with a tiny file. Real data is messy; partitioning makes the mess visible.
- Many folders × several files each = the **small files problem**: lots of tiny files are slow to list and open, especially on S3.

> **Coming from Hive/Impala:** the same small-files problem you know from HDFS — there it costs NameNode memory; on S3 it costs one slow request per file.

**What this cell does**
- Reads the partitioned folder back. Spark sees the `pickup_date=...` folder names and turns them into a `pickup_date` column.
- `f"{PLAIN}/trips_by_day"` is an **f-string**: `{PLAIN}` is replaced by the value of the `PLAIN` variable.
- Prints two plans: one filtering on the original timestamp column, one on the partition column. Compare the `PartitionFilters:` part of each.

In [ ]:
part = spark.read.parquet(f"{PLAIN}/trips_by_day")

print("=== filter on the TIMESTAMP column ===")
part.filter("tpep_pickup_datetime >= '2024-01-15' AND tpep_pickup_datetime < '2024-01-16'").explain()

print("=== filter on the PARTITION column ===")
part.filter("pickup_date = '2024-01-15'").explain()

**What to notice** — look for `PartitionFilters` in each plan:
- Filtering on `tpep_pickup_datetime` → `PartitionFilters: []`. Spark reads **every folder**, even though the data is partitioned by day.
- Filtering on `pickup_date` → `PartitionFilters: [... (pickup_date = 2024-01-15)]`. Spark reads **one folder**.

Every user has to know about the extra `pickup_date` column and remember to filter on it. Most won't. Iceberg's **hidden partitioning** (notebook 02) removes this trap.

> **Coming from Hive/Impala:** the classic Hive trap — `WHERE event_time >= '2024-01-15'` scans every partition; only `WHERE dt = '2024-01-15'` prunes. Same behaviour here, for the same reason: the folder name is the only thing the engine can prune on.

## 6. What plain Parquet can't do

Three everyday needs that a folder of Parquet files can't handle safely.

### 6a. Delete bad rows
There are trips with negative fares. Let's try to delete them.

**What this cell does**
- First counts the rows with negative fares.
- `` DELETE FROM parquet.`folder` WHERE ... `` then tries a SQL delete directly on the Parquet folder.
- `try:` / `except Exception as e:` = "try this; if it fails, don't stop the notebook — run the `except` part instead with the error in `e`". We *expect* this to fail.
- The `re.sub(...)` line just shortens Spark's long error message (it lists every column) so it's readable.

In [ ]:
print("negative fares:", df.filter("fare_amount < 0").count())
import re
try:
    spark.sql(f"DELETE FROM parquet.`{PLAIN}/trips_jan` WHERE fare_amount < 0")
except Exception as e:
    reason = next(l for l in str(e).splitlines() if "Exception" in l)
    print("\nDELETE failed:", re.sub(r"Relation \[.*\]", "Relation [...]", reason).strip(": "))

**What to notice:** Spark fails with an internal error — it has no way to delete rows from a plain Parquet folder ("table relation ... parquet"). Parquet files are **immutable** and a folder has no transaction log, so there is no `DELETE`, `UPDATE` or `MERGE`. The only way is: read everything, filter, write a *new* folder, then swap folders — and hope no one reads in between.

> **Coming from Hive/Impala:** Hive can `DELETE` only on ACID tables (transactional ORC, with background compaction); Impala can't delete from Parquet tables at all. The usual Hadoop workaround is `INSERT OVERWRITE` of each affected partition with the rows you want to keep — the same read-filter-rewrite, done by hand.

### 6b. Change the schema
Suppose the publisher renames `Airport_fee` to `airport_surcharge` starting in February. We append February to the same folder.

**What this cell does**
- Reads February and renames one column: `.withColumnRenamed("Airport_fee", "airport_surcharge")` = `SELECT ..., Airport_fee AS airport_surcharge`.
- `.write.mode("append")` **adds** February's files to the January folder (like `INSERT INTO`), without touching January's files.
- `.option("mergeSchema", "true")` asks Spark to combine the schemas of all files in the folder, instead of trusting just one file.
- The `groupBy(month)...agg(count(...))` counts non-NULL values of each airport column per month (`count(col)` skips NULLs, like in SQL).

In [ ]:
feb = (spark.read.parquet("/home/lab/work/data/raw/yellow_tripdata_2024-02.parquet")
            .withColumnRenamed("Airport_fee", "airport_surcharge"))
feb.write.mode("append").parquet(f"{PLAIN}/trips_jan")

mixed = spark.read.option("mergeSchema", "true").parquet(f"{PLAIN}/trips_jan")
print([c for c in mixed.columns if "irport" in c])   # columns with "airport" in the name
(mixed.groupBy(F.month("tpep_pickup_datetime").alias("month"))
      .agg(F.count("Airport_fee").alias("Airport_fee_values"),
           F.count("airport_surcharge").alias("airport_surcharge_values"))
      .filter("month IN (1, 2)").orderBy("month").show())

**What to notice:** the table now has **two** columns for the same thing. January's values are almost all in `Airport_fee`, February's in `airport_surcharge` (`count()` skips NULLs). The few "crossovers" are trips dated in the other month — each monthly file contains a handful of late or early records. Plain Parquet matches columns **by name**, so a rename looks like "drop one column, add another". Without `mergeSchema`, Spark silently uses one file's schema and hides a column entirely.

> Side note: Spark matches column names **case-insensitively** by default (`spark.sql.caseSensitive=false`), so renaming only the case (`Airport_fee` → `airport_fee`) would *not* split the column. Real renames do.

> **Coming from Hive/Impala:** you may have seen this as "the column is NULL for old partitions after `ALTER TABLE ... CHANGE COLUMN`". Hive matches Parquet columns **by name**, so renamed columns read as NULL in old files; Impala matches **by position** by default, so dropping or reordering columns shifts values into the wrong column. Either way, the files don't know about the rename.

### 6c. History and safe writes
- `mode("overwrite")` **deletes the old files first**, then writes new ones. If the job dies halfway, the folder is left half-written, and the old data is gone.
- A reader that lists the folder *during* a write can see a mix of old and new files.
- There's no "what did this table look like yesterday?" — no versions, no undo.

> **Coming from Hive/Impala:** this is `INSERT OVERWRITE` on HDFS. The closest thing to history is an HDFS snapshot of the whole directory, set up by an admin.

## Summary

| Need | Plain Parquet folder | Iceberg table (notebook 02) |
|---|---|---|
| Read only needed columns | ✔ columnar | ✔ (it *is* Parquet underneath) |
| Skip data using min/max | only if the writer added stats | ✔ stats kept in metadata for every file |
| Partition pruning | only if users filter on the extra column | ✔ hidden partitioning: filter on the real column |
| DELETE / UPDATE / MERGE | ✘ rewrite folders by hand | ✔ SQL commands, atomic |
| Rename a column | ✘ looks like drop + add | ✔ columns tracked by ID, metadata-only |
| Safe concurrent reads & writes | ✘ readers can see half a write | ✔ snapshot isolation |
| History / time travel / undo | ✘ | ✔ every write is a snapshot |

## Exercises
1. Using the table from section 1, which column has the **best** compression ratio, and why do you think so?
2. Write `df` partitioned by `payment_type` to `f"{PLAIN}/trips_by_payment"`. How many folders and files do you get? Is this a good partition column? (Hint: think about how queries filter, and how evenly data spreads.)
3. How many January rows have a pickup time **outside** January 2024? Write the query with the DataFrame API.

Next: **`02_iceberg_first_table.ipynb`**.